# Day 8b — Feature-fusion classifier (a third combination attempt)

Day 8's two attempts combined the classifier and the graph *after* the classifier was already trained and frozen (score-level fusion). Research on this exact dataset (see `docs/hybrid-merge.md` for citations) points to a different, more common pattern: **feature-level fusion** — feed graph-derived features (cluster category, cluster size) into the classifier as ordinary input columns, and retrain, so the model can learn the right way to weigh them itself instead of a hand-written multiply/threshold rule.

**This produces a new classifier version (v3), separate from the Day 4 locked classifier.** The Day 4 result stands exactly as reported; this is a new, independent experiment.

**No new leakage risk**: cluster *category* (unlike Day 8's category *weight*) is computed purely from graph structure — it never looks at `isFraud` — so it's safe to use as a feature on train, holdout, or any future transaction, with no train-only recomputation needed.

In [1]:
import sys, json
sys.path.append('..')

import numpy as np
import pandas as pd

from src.data import load_merged_train
from src.features import get_feature_lists, build_preprocessor
from src.split import apply_locked_split
from src.graph import load_graph
from src.model import build_xgb_pipeline
from src.cost import total_cost, find_optimal_threshold, DEFAULT_REVIEW_COST
from src.hybrid import assign_cluster_categories

RANDOM_STATE = 42

## 1. Add ring features to every transaction

In [2]:
train_full = load_merged_train()

identity_graph = load_graph('../data/processed/identity_graph.pkl')
behavioral_graph = load_graph('../data/processed/behavioral_graph.pkl')
with open('../data/processed/cluster_partition.json') as f:
    partition = {int(k): v for k, v in json.load(f).items()}

ring_categories = assign_cluster_categories(train_full['TransactionID'], partition, identity_graph, behavioral_graph)
cluster_sizes = pd.Series(partition).value_counts()

train_full = train_full.set_index('TransactionID', drop=False)
train_full['ring_category'] = ring_categories
train_full['ring_cluster_size'] = train_full['TransactionID'].map(partition).map(cluster_sizes)
train_full = train_full.reset_index(drop=True)

print(train_full['ring_category'].value_counts())
print('\nring_cluster_size summary:')
print(train_full['ring_cluster_size'].describe())

ring_category
both               285483
isolated           224560
behavioral_only     45662
identity_only       34835
Name: count, dtype: int64

ring_cluster_size summary:
count    590540.000000
mean        149.816206
std         409.201396
min           1.000000
25%           1.000000
50%           4.000000
75%         217.000000
max        4187.000000
Name: ring_cluster_size, dtype: float64


## 2. Feature lists: base features + the two new ring features

In [3]:
numeric_features, categorical_features = get_feature_lists(train_full)
numeric_features = numeric_features + ['ring_cluster_size']
categorical_features = categorical_features + ['ring_category']

print(f'{len(numeric_features)} numeric features (incl. ring_cluster_size)')
print(f'{len(categorical_features)} categorical features (incl. ring_category)')

389 numeric features (incl. ring_cluster_size)
16 categorical features (incl. ring_category)


## 3. Split, train, evaluate — same locked holdout as every other classifier version

In [4]:
train_df, holdout_df = apply_locked_split(train_full)

X_train = train_df[numeric_features + categorical_features]
y_train = train_df['isFraud']
X_holdout = holdout_df[numeric_features + categorical_features]
y_holdout = holdout_df['isFraud']
amounts_holdout = holdout_df['TransactionAmt']

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model_v3 = build_xgb_pipeline(numeric_features, categorical_features, scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE)
model_v3.fit(X_train, y_train)
print('Fitted v3 (feature-fusion) classifier.')

Fitted v3 (feature-fusion) classifier.


In [5]:
y_proba_v3 = model_v3.predict_proba(X_holdout)[:, 1]

v3_threshold, v3_cost, _ = find_optimal_threshold(y_holdout, y_proba_v3, amounts_holdout, review_cost=DEFAULT_REVIEW_COST)

with open('../results/classifier_final_metrics.json') as f:
    locked_v2_metrics = json.load(f)
v2_cost = locked_v2_metrics['total_cost_rs']

print(f'v3 (feature-fusion) cost-optimal threshold: {v3_threshold:.2f}')
print(f'v3 cost: Rs {v3_cost:,.0f}')
print(f'v2 (locked, no ring features) cost: Rs {v2_cost:,.0f}')
print(f'Difference: Rs {v2_cost - v3_cost:,.0f} ({"v3 better" if v3_cost < v2_cost else "v2 better"})')

v3 (feature-fusion) cost-optimal threshold: 0.87
v3 cost: Rs 333,872
v2 (locked, no ring features) cost: Rs 337,421
Difference: Rs 3,549 (v3 better)


In [6]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, confusion_matrix

y_pred_v3 = (y_proba_v3 >= v3_threshold).astype(int)
precision_v3 = precision_score(y_holdout, y_pred_v3)
recall_v3 = recall_score(y_holdout, y_pred_v3)
f1_v3 = f1_score(y_holdout, y_pred_v3)
pr_auc_v3 = average_precision_score(y_holdout, y_proba_v3)
cm_v3 = confusion_matrix(y_holdout, y_pred_v3)

print(f'Precision: {precision_v3:.4f} (v2 locked: {locked_v2_metrics["precision"]:.4f})')
print(f'Recall:    {recall_v3:.4f} (v2 locked: {locked_v2_metrics["recall"]:.4f})')
print(f'F1:        {f1_v3:.4f} (v2 locked: {locked_v2_metrics["f1"]:.4f})')
print(f'PR-AUC:    {pr_auc_v3:.4f}')
print('\nConfusion matrix:')
print(cm_v3)

Precision: 0.8350 (v2 locked: 0.7582)
Recall:    0.6136 (v2 locked: 0.6593)
F1:        0.7074 (v2 locked: 0.7053)
PR-AUC:    0.7566

Confusion matrix:
[[113474    501]
 [  1597   2536]]


## 4. Did the ring features actually matter to the model?

In [7]:
preprocessor_v3 = model_v3.named_steps['prep']
classifier_v3 = model_v3.named_steps['clf']
feature_names_v3 = preprocessor_v3.get_feature_names_out()

importances = pd.Series(classifier_v3.feature_importances_, index=feature_names_v3).sort_values(ascending=False)

ring_feature_importances = importances[importances.index.str.contains('ring_')]
print('Ring feature importances (and their rank out of', len(importances), 'total features):')
for name, value in ring_feature_importances.items():
    rank = importances.index.get_loc(name) + 1
    print(f'  {name}: importance={value:.5f}, rank={rank}')

print('\nTop 10 features overall:')
print(importances.head(10))

Ring feature importances (and their rank out of 560 total features):
  cat__ring_category_identity_only: importance=0.00106, rank=190
  cat__ring_category_behavioral_only: importance=0.00095, rank=223
  num__ring_cluster_size: importance=0.00082, rank=260
  cat__ring_category_both: importance=0.00065, rank=303
  cat__ring_category_isolated: importance=0.00000, rank=484

Top 10 features overall:
num__V258           0.138148
num__V70            0.098308
num__V201           0.072500
num__V91            0.044627
num__V294           0.023451
num__V295           0.023225
cat__ProductCD_C    0.019163
cat__card6_debit    0.015369
num__V102           0.011644
num__V312           0.010630
dtype: float32


## Save results

In [8]:
feature_fusion_results = {
    'model': 'XGBoost v3 (feature-fusion: ring_category + ring_cluster_size added as features)',
    'threshold': v3_threshold,
    'precision': precision_v3,
    'recall': recall_v3,
    'f1': f1_v3,
    'pr_auc': pr_auc_v3,
    'confusion_matrix': cm_v3.tolist(),
    'total_cost_rs': v3_cost,
    'v2_locked_cost_rs': v2_cost,
    'difference_rs': v2_cost - v3_cost,
    'ring_feature_importances': ring_feature_importances.to_dict(),
    'ring_feature_count_in_top_10': int(importances.head(10).index.str.contains('ring_').sum()),
}

with open('../results/feature_fusion_metrics.json', 'w') as f:
    json.dump(feature_fusion_results, f, indent=2, default=str)

print('Saved to results/feature_fusion_metrics.json')

Saved to results/feature_fusion_metrics.json


## Takeaway

_Fill in after running: whether v3 beats the locked v2 classifier on cost, where the ring features ranked in importance, and an honest verdict on whether feature-fusion was worth it compared to Day 8's two score-level attempts._